Split the source .csv files in different batches to simulate continuous ingestion with aoutoloader

## Cust_info

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(
        "/Volumes/data_lakehouse_databricks/bronze/bronze_vol/source_crm/cust_info.csv"
    )
)

print(df.count())

w = Window.orderBy(F.monotonically_increasing_id())

df_batches = (
    df
    .withColumn(
        "_row_number",
        F.row_number().over(w)
    )
)

In [0]:
batch_1 = (
    df_batches
    .filter(F.col("_row_number") <= 6000)
    .drop("_row_number")
)

batch_2 = (
    df_batches
    .filter(
        (F.col("_row_number") > 6000)
        & (F.col("_row_number") <= 12000)
    )
    .drop("_row_number")
)

batch_3 = (
    df_batches
    .filter(F.col("_row_number") > 12000)
    .drop("_row_number")
)

In [0]:
base = (
    "/Volumes/data_lakehouse_databricks/bronze/landing_vol/generated_cust/"
)

batch_1.coalesce(1).write.mode("overwrite").option(
    "header", True
).csv(f"{base}/batch_1")

batch_2.coalesce(1).write.mode("overwrite").option(
    "header", True
).csv(f"{base}/batch_2")

batch_3.coalesce(1).write.mode("overwrite").option(
    "header", True
).csv(f"{base}/batch_3")

## Sales_details

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(
        "/Volumes/data_lakehouse_databricks/bronze/bronze_vol/source_crm/sales_details.csv"
    )
)

print(df.count())

w = Window.orderBy(F.monotonically_increasing_id())

df_batches = (
    df
    .withColumn(
        "_row_number",
        F.row_number().over(w)
    )
)

In [0]:
batch_1 = (
    df_batches
    .filter(F.col("_row_number") <= 20000)
    .drop("_row_number")
)

batch_2 = (
    df_batches
    .filter(
        (F.col("_row_number") > 20000)
        & (F.col("_row_number") <= 40000)
    )
    .drop("_row_number")
)

batch_3 = (
    df_batches
    .filter(F.col("_row_number") > 40000)
    .drop("_row_number")
)

In [0]:
base = (
    "/Volumes/data_lakehouse_databricks/bronze/landing_vol/generated_sales/"
)

batch_1.coalesce(1).write.mode("overwrite").option(
    "header", True
).csv(f"{base}/batch_1")

batch_2.coalesce(1).write.mode("overwrite").option(
    "header", True
).csv(f"{base}/batch_2")

batch_3.coalesce(1).write.mode("overwrite").option(
    "header", True
).csv(f"{base}/batch_3")